In [1]:
import numpy as np
from scipy.integrate import quad
from scipy.linalg import solve_continuous_lyapunov

# Rates in s^-1. Set chi = omega0 * beta (per-photon frequency shift).
gamma = 0.1
chi = 1.0
n_modes = 10
n_c = 100.0          # Per-mode reference occupation
nbar = 110.0          # Target mean occupation per mode; choose nbar > n_c

if n_modes < 1 or gamma <= 0 or nbar <= n_c:
    raise ValueError("Use n_modes >= 1, gamma > 0, and nbar > n_c.")

# This model has no direct four-wave mixing or photon exchange between modes.
# Every mode sees the same Kerr detuning, determined by the TOTAL photon number:
#   delta = chi * (sum_j n_j - n_modes * n_c).
# Thus n_c is the per-mode reference, and the common resonance center is
# n_modes * n_c. For a single global reference instead, use n_c_total = n_c.
nbar_modes = np.full(n_modes, nbar, dtype=float)
nc_modes = np.full(n_modes, n_c, dtype=float)
n_total = float(np.sum(nbar_modes))
n_c_total = float(np.sum(nc_modes))
delta = chi * (n_total - n_c_total)

# Choose identical real coherent drives to give the requested occupation per mode.
# Steady state: alpha_j = sqrt(2 gamma) s0_j / (gamma + i delta).
s0_modes = np.sqrt(nbar_modes * (gamma**2 + delta**2) / (2 * gamma))
alpha = np.sqrt(2 * gamma) * s0_modes / (gamma + 1j * delta)

# Linearize in the Nambu vector z = (delta_a_1,...,delta_a_N,
# delta_a_1^dagger,...,delta_a_N^dagger).
# The shared intensity fluctuation is delta_N = sum_j(alpha_j* delta_a_j
#                                      + alpha_j delta_a_j^dagger).
I_N = np.eye(n_modes)
A = np.zeros((2 * n_modes, 2 * n_modes), dtype=complex)
A[:n_modes, :n_modes] = (
    (-gamma - 1j * delta) * I_N
    - 1j * chi * np.outer(alpha, np.conj(alpha))
)
A[:n_modes, n_modes:] = -1j * chi * np.outer(alpha, alpha)
A[n_modes:, :n_modes] = 1j * chi * np.outer(np.conj(alpha), np.conj(alpha))
A[n_modes:, n_modes:] = (
    (-gamma + 1j * delta) * I_N
    + 1j * chi * np.outer(np.conj(alpha), alpha)
)

# Convert to real quadratures q = (delta_X_1,...,delta_X_N,
#                                  delta_Y_1,...,delta_Y_N),
# with a_j = (X_j + i Y_j)/sqrt(2).
T = np.block([
    [I_N, I_N],
    [-1j * I_N, 1j * I_N],
]) / np.sqrt(2)
Aq_complex = T @ A @ T.conj().T
assert np.max(np.abs(Aq_complex.imag)) < 1e-10 * np.max(np.abs(Aq_complex.real))
Aq = Aq_complex.real

eigs = np.linalg.eigvals(Aq)
if np.max(np.real(eigs)) >= 0:
    raise ValueError(f"Linearized steady state is unstable: eigenvalues={eigs}")

# Independent vacuum inputs for the modes. Each Langevin noise amplitude is
# sqrt(2 gamma), so the symmetrized quadrature diffusion matrix is D = gamma I.
D = gamma * np.eye(2 * n_modes)
I_2N = np.eye(2 * n_modes)

# Rows of Q are the linearized photon-number gradients:
#   delta_n_j ~= alpha_j* delta_a_j + alpha_j delta_a_j^dagger.
Q = np.zeros((n_modes, 2 * n_modes), dtype=float)
mode_indices = np.arange(n_modes)
Q[mode_indices, mode_indices] = np.sqrt(2) * alpha.real
Q[mode_indices, n_modes + mode_indices] = np.sqrt(2) * alpha.imag


def S_number_matrix(Omega):
    """Real part of the symmetrized two-sided number cross-spectrum matrix.

    Omega is an angular analysis frequency in rad/s. Diagonal entries are
    S_{n_j n_j}(Omega); off-diagonal entries are inter-mode number spectra.
    """
    H = np.linalg.solve(-1j * Omega * I_2N - Aq, I_2N)
    S_q = H @ D @ H.conj().T
    return np.real(Q @ S_q @ Q.T)


# Integrate each even spectrum. Break the integration at every resonance and at
# multiples of the narrowest linewidth around it (clustered float duplicates removed),
# and map the infinite tail Omega = L/u onto (0, 1].
peak_frequencies = np.unique(np.round(np.abs(np.imag(eigs)), 9))
width_min = float(np.min(np.maximum(-np.real(eigs), 1e-14)))
omega_max = max(
    50 * gamma,
    float(np.max(np.abs(np.imag(eigs))) + 50 * np.max(-np.real(eigs))),
)
points = {0.0}
for peak in peak_frequencies:
    for m in (-200, -50, -10, -2, -0.5, 0, 0.5, 2, 10, 50, 200):
        x = peak + m * width_min
        if 0 < x < 20 * omega_max:
            points.add(round(float(x), 9))
edges = np.array(sorted(points) + [20 * omega_max])

var_spectrum = np.empty(n_modes, dtype=float)
for j in range(n_modes):
    def mode_spectrum(Omega):
        return float(S_number_matrix(Omega)[j, j])

    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        total += quad(mode_spectrum, lo, hi, epsabs=1e-12, epsrel=1e-12, limit=500)[0]
    L_tail = edges[-1]
    total += quad(lambda u: mode_spectrum(L_tail / u) * L_tail / u**2, 0.0, 1.0,
                  epsabs=1e-12, epsrel=1e-12, limit=500)[0]
    # Var(n_j) = integral_{-inf}^{inf} S_jj(Omega) dOmega / (2 pi).
    var_spectrum[j] = total / np.pi
F_spectrum = var_spectrum / nbar_modes

# Independent covariance check: Aq V + V Aq^T + D = 0.
V = solve_continuous_lyapunov(Aq, -D)
V = 0.5 * (V + V.T)
number_covariance = Q @ V @ Q.T
number_covariance = 0.5 * (number_covariance + number_covariance.T)
var_covariance = np.diag(number_covariance)
F_covariance = var_covariance / nbar_modes

# Total-number Fano factor includes all inter-mode covariances.
var_total = float(np.sum(number_covariance))
F_total = var_total / n_total

# Closed-form symmetric-mode check. With identical drives and Kerr pulls, only
# the bright (equal-phase) collective mode is squeezed; the other N-1 modes are
# dark vacuum fluctuations. This predicts the per-line Fano factor as well.
denom_bright = (
    gamma**2
    + chi**2 * (n_total - n_c_total) * (3 * n_total - n_c_total)
)
F_bright_closed = (
    1
    - chi**2 * n_total * (n_total - n_c_total) / denom_bright
)
F_line_closed = 1 + (F_bright_closed - 1) / n_modes

print(f"number of modes                  = {n_modes}")
print(f"mean photon number per mode      = {np.abs(alpha)**2}")
print(f"common Kerr detuning [rad/s]     = {delta:.6g}")
print(f"drive amplitude per mode |s0_j|  = {s0_modes}")
print(f"stable                           = {np.max(np.real(eigs)) < 0}")
print(f"Fano factors, spectrum           = {F_spectrum}")
print(f"Fano factors, covariance         = {F_covariance}")
print(f"Fano factor of total photon number = {F_total:.6g}")
print(f"symmetric-mode check, per line   = {F_line_closed:.6g}")
print(f"symmetric-mode check, total      = {F_bright_closed:.6g}")
print("inter-mode number covariance matrix =")
print(number_covariance)

number of modes                  = 10
mean photon number per mode      = [110. 110. 110. 110. 110. 110. 110. 110. 110. 110.]
common Kerr detuning [rad/s]     = 100
drive amplitude per mode |s0_j|  = [2345.20905252 2345.20905252 2345.20905252 2345.20905252 2345.20905252
 2345.20905252 2345.20905252 2345.20905252 2345.20905252 2345.20905252]
stable                           = True
Fano factors, spectrum           = [0.95217392 0.95217392 0.95217392 0.95217392 0.95217392 0.95217392
 0.95217392 0.95217392 0.95217392 0.95217392]
Fano factors, covariance         = [0.95217392 0.95217392 0.95217392 0.95217392 0.95217392 0.95217392
 0.95217392 0.95217392 0.95217392 0.95217392]
Fano factor of total photon number = 0.521739
symmetric-mode check, per line   = 0.952174
symmetric-mode check, total      = 0.521739
inter-mode number covariance matrix =
[[104.73913066  -5.26086934  -5.26086934  -5.26086934  -5.26086934
   -5.26086934  -5.26086934  -5.26086934  -5.26086934  -5.26086934]
 [ -5.26086934 